In [6]:
import os
import pandas as pd

In [ ]:
## Stack data downloaded from https://www.eia.gov/electricity/wholesalemarkets/data.php?rto=caiso into one file for hourly day ahead prices 
## Same code also works for load forecast data, and renewable generation data

# Define the base directory for Day ahead price data
base_path = "Data/unproc/"

# Define available years
years = [2020, 2021, 2022, 2023, 2024]

# Initialize a list to store dataframes
data_list = []

# Loop through each year and process data
for year in years:
    file_path = os.path.join(base_path, f"caiso_lmp_da_hr_zones_{year}.csv")

    if os.path.exists(file_path):  # Check if file exists before loading
        df = pd.read_csv(file_path, skiprows=3, delimiter=",")
        df["year"] = year  # Add a year column
        data_list.append(df)
        print(f"✅ Loaded: {file_path}")
    else:
        print(f"❌ Missing file: {file_path}")

# Concatenate all available data
if data_list:
    stacked_data = pd.concat(data_list, ignore_index=True)

    # Save the stacked dataset
    output_file_path = "Data/preprocessed/stacked_lmp_da_hr_data.csv"
    stacked_data.to_csv(output_file_path, index=False)
    print(f"✅ Stacked data saved to: {output_file_path}")

else:
    print("❌ No files were found. Please check the directory.")


In [ ]:
## Stack real time price data downloaded from https://www.eia.gov/electricity/wholesalemarkets/data.php?rto=caiso into one file


# Define the base directory
base_path = "Data/unproc/"

# Define the range of years and quarters
years = range(2020, 2026)  # Covers 2020 to 2025
quarters = ["Q1", "Q2", "Q3", "Q4"]

# Initialize a list to store dataframes
price_data_list = []

# Loop through each year and quarter to load and append preprocessed LMP data
for year in years:
    for quarter in quarters:
        quarter_str = f"{year}{quarter}"
        if quarter_str < "2020Q4" or quarter_str > "2025Q1":  # Skip out-of-range quarters
            continue

        # Construct the correct file path
        file_path = f"/content/drive/MyDrive/PowerPrices/Data/preprocessed/{year}/preprocessed_lmp_rt_{year}{quarter}.csv"

        # Load the file
        df = pd.read_csv(file_path)
        df["quarter"] = quarter_str  # Add a column to identify the quarter
        price_data_list.append(df)
        print(f"✅ Loaded: {file_path}")

# Concatenate all LMP price dataframes into a single dataframe
stacked_lmp_data = pd.concat(price_data_list, ignore_index=True)

# Define the output path for the stacked LMP data
output_file_path = "/content/drive/MyDrive/PowerPrices/Data/preprocessed/stacked_lmp_price_data.csv"

# Save the stacked dataframe as a CSV file
stacked_lmp_data.to_csv(output_file_path, index=False)
print(f"✅ Stacked LMP price data saved to: {output_file_path}")

# # Display the stacked dataframe
# import ace_tools as tools
# tools.display_dataframe_to_user(name="Stacked LMP Price Data", dataframe=stacked_lmp_data)


In [7]:
# Remove duplicate times by replacing with mean value and insert missing times

def process_begin_times(df):
    df['begin_time'] = pd.to_datetime(df['begin_time'])
    df.set_index('begin_time', inplace=True)
    df = df.groupby(df.index).mean()
    dates = pd.date_range(start=df.index.min(), end=df.index.max(), freq='15min')
    df = df.reindex(dates)
    df.reset_index(inplace=True)
    df.rename(columns={'index':'begin_time'}, inplace=True)
    return df 

The following code prepares exogenous variables for price forecasting.

In [26]:
## Hourly load data downloaded from gridstatus.io
## Forward fill to get values at 15 min intervals, since load at end of hour is not known

load_data = pd.read_csv('Data/load_1hr_PGE.csv', usecols=['interval_start_local','load'])
load_data.rename(columns={'interval_start_local':'begin_time', 'load':'load_actual'}, inplace=True)
load_data['begin_time'] = pd.to_datetime(load_data['begin_time']).apply(lambda x : x.tz_localize(None))
load_data = process_begin_times(load_data)
load_data.ffill(inplace=True)
load_data

/var/folders/y1/qbgxvx6j1vj5ydy42yxx7fsm0000gn/T/ipykernel_3675/3763765480.py:6: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  load_data['begin_time'] = pd.to_datetime(load_data['begin_time']).apply(lambda x : x.tz_localize(None))


,begin_time,load_actual
0,2020-01-01 00:00:00,10206.0
1,2020-01-01 00:15:00,10206.0
2,2020-01-01 00:30:00,10206.0
3,2020-01-01 00:45:00,10206.0
4,2020-01-01 01:00:00,9866.0
...,...,...
175384,2024-12-31 22:00:00,10226.0
175385,2024-12-31 22:15:00,10226.0
175386,2024-12-31 22:30:00,10226.0
175387,2024-12-31 22:45:00,10226.0


In [27]:
## Hourly Load forecast data downloaded from https://www.eia.gov/electricity/wholesalemarkets/data.php?rto=caiso
## Linearly interpolate to get forecasts at 15 min intervals, can do this since forecasts are released the previous day

load_forecast = pd.read_csv('Data/preprocessed/stacked_load_forecast_data.csv', usecols=['Local Timestamp Pacific Time (Interval Beginning)', 'Pacific Gas and Electric Forecast Load (MW)'])
load_forecast.rename(columns={'Local Timestamp Pacific Time (Interval Beginning)':'begin_time', 'Pacific Gas and Electric Forecast Load (MW)':'load_forecast'}, inplace=True)
load_forecast=process_begin_times(load_forecast)
load_forecast.interpolate(inplace=True)
load_forecast
load_all = pd.merge(load_data, load_forecast, on='begin_time', how='inner')
load_all

,begin_time,load_actual,load_forecast
0,2021-03-12 00:00:00,9914.0,10102.7200
1,2021-03-12 00:15:00,9914.0,10044.1825
2,2021-03-12 00:30:00,9914.0,9985.6450
3,2021-03-12 00:45:00,9914.0,9927.1075
4,2021-03-12 01:00:00,9827.0,9868.5700
...,...,...,...
133528,2024-12-31 22:00:00,10226.0,10831.4600
133529,2024-12-31 22:15:00,10226.0,10682.2425
133530,2024-12-31 22:30:00,10226.0,10533.0250
133531,2024-12-31 22:45:00,10226.0,10383.8075


In [28]:
## Hourly solar/wind generation forecast data downloaded from gridstatus.io
## Linearly interpolate to get forecasts at 15 min intervals, can do this since all forecasts are released by the previous day

renforecast = pd.read_csv('Data/solarwind_forecast_1hr_NP15.csv', usecols=['interval_start_local', 'solar_mw', 'wind_mw'])
renforecast.rename(columns={'interval_start_local':'begin_time'}, inplace=True)
renforecast = renforecast.groupby('begin_time').last() #Pick the most recent forecast when multiple forecasts for the same time are available
renforecast.reset_index(inplace=True)
renforecast.rename(columns={'index':'begin_time', 'solar_mw':'solar_forecast', 'wind_mw':'wind_forecast'}, inplace=True)
renforecast['begin_time'] = pd.to_datetime(renforecast['begin_time']).apply(lambda x : x.tz_localize(None))
renforecast = process_begin_times(renforecast)
renforecast.interpolate(inplace=True)
renforecast

/var/folders/y1/qbgxvx6j1vj5ydy42yxx7fsm0000gn/T/ipykernel_3675/1047603222.py:9: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  renforecast['begin_time'] = pd.to_datetime(renforecast['begin_time']).apply(lambda x : x.tz_localize(None))


,begin_time,solar_forecast,wind_forecast
0,2021-01-01 00:00:00,0.0,99.2700
1,2021-01-01 00:15:00,0.0,91.5825
2,2021-01-01 00:30:00,0.0,83.8950
3,2021-01-01 00:45:00,0.0,76.2075
4,2021-01-01 01:00:00,0.0,68.5200
...,...,...,...
148120,2025-03-23 22:00:00,0.0,184.6500
148121,2025-03-23 22:15:00,0.0,182.9950
148122,2025-03-23 22:30:00,0.0,181.3400
148123,2025-03-23 22:45:00,0.0,179.6850


In [29]:
## Actual renewable electricity generation data downloaded from https://www.eia.gov/electricity/wholesalemarkets/data.php?rto=caiso
## Forward fill to get values at 15 min intervals, since generation at end of hour is not known

ren_gen_actual = pd.read_csv('Data/preprocessed/stacked_gen_ren_hr_data.csv', usecols=['Local Timestamp Pacific Time (Interval Beginning)','NP15 Solar Generation (MW)','NP15 Wind Generation (MW)'])
ren_gen_actual.rename(columns={'Local Timestamp Pacific Time (Interval Beginning)':'begin_time', 'NP15 Solar Generation (MW)':'solar_actual', 'NP15 Wind Generation (MW)':'wind_actual'}, inplace=True)
ren_gen_actual = process_begin_times(ren_gen_actual)
ren_gen_actual.ffill(inplace=True)
ren_all = pd.merge(renforecast, ren_gen_actual, on='begin_time', how='inner')
ren_all

,begin_time,solar_forecast,wind_forecast,solar_actual,wind_actual
0,2021-01-01 00:00:00,0.0,99.2700,-3.88458,179.27404
1,2021-01-01 00:15:00,0.0,91.5825,-3.88458,179.27404
2,2021-01-01 00:30:00,0.0,83.8950,-3.88458,179.27404
3,2021-01-01 00:45:00,0.0,76.2075,-3.88458,179.27404
4,2021-01-01 01:00:00,0.0,68.5200,-3.91677,97.43401
...,...,...,...,...,...
140248,2024-12-31 22:00:00,0.0,77.4600,-6.26466,54.79276
140249,2024-12-31 22:15:00,0.0,80.2975,-6.26466,54.79276
140250,2024-12-31 22:30:00,0.0,83.1350,-6.26466,54.79276
140251,2024-12-31 22:45:00,0.0,85.9725,-6.26466,54.79276


In [37]:
## Henry Hub natural gas spot prices downloaded from https://www.eia.gov/dnav/ng/hist/rngwhhdm.htm
## This code puts prices in a series with time stamps at one hour intervals

natural_gas_price = pd.read_csv('Data/Henry_Hub_Natural_Gas_Spot_Price.csv', skiprows=4)
natural_gas_price.rename(columns={'Henry Hub Natural Gas Spot Price Dollars per Million Btu':'nat_gas_price'}, inplace=True)
natural_gas_price['Day'] = pd.to_datetime(natural_gas_price['Day'])
natural_gas_price = natural_gas_price[::-1]
natural_gas_price.set_index('Day', inplace=True)
dates = pd.date_range(start=natural_gas_price.index.min(), end=natural_gas_price.index.max(), freq='15min')
natural_gas_price = natural_gas_price.reindex(dates)
natural_gas_price['nat_gas_price'] = natural_gas_price['nat_gas_price'].ffill()
natural_gas_price.reset_index(inplace=True)
natural_gas_price.rename(columns={'index':'begin_time'}, inplace=True)
natural_gas_price.to_csv('Data/preprocessed/nat_gas_price.csv', index=False)
natural_gas_price

,begin_time,nat_gas_price
0,1997-01-07 00:00:00,3.82
1,1997-01-07 00:15:00,3.82
2,1997-01-07 00:30:00,3.82
3,1997-01-07 00:45:00,3.82
4,1997-01-07 01:00:00,3.82
...,...,...
986396,2025-02-23 23:00:00,4.43
986397,2025-02-23 23:15:00,4.43
986398,2025-02-23 23:30:00,4.43
986399,2025-02-23 23:45:00,4.43


In [38]:
## Combine to form one exogenous variables dataframe

load_gen = pd.merge(load_all, ren_all, on='begin_time', how='inner')
nat_gas_price = pd.read_csv('Data/preprocessed/nat_gas_price.csv')
nat_gas_price['begin_time'] = pd.to_datetime(nat_gas_price['begin_time'])
exog = pd.merge(load_gen, nat_gas_price, how='inner', on='begin_time')
exog.to_csv('Data/preprocessed/NP15_exog.csv', index=False)
exog

,begin_time,load_actual,load_forecast,solar_forecast,wind_forecast,solar_actual,wind_actual,nat_gas_price
0,2021-03-12 00:00:00,9914.0,10102.7200,0.0,327.9700,-2.69832,526.78168,2.65
1,2021-03-12 00:15:00,9914.0,10044.1825,0.0,310.9625,-2.69832,526.78168,2.65
2,2021-03-12 00:30:00,9914.0,9985.6450,0.0,293.9550,-2.69832,526.78168,2.65
3,2021-03-12 00:45:00,9914.0,9927.1075,0.0,276.9475,-2.69832,526.78168,2.65
4,2021-03-12 01:00:00,9827.0,9868.5700,0.0,259.9400,-2.66009,405.97186,2.65
...,...,...,...,...,...,...,...,...
133528,2024-12-31 22:00:00,10226.0,10831.4600,0.0,77.4600,-6.26466,54.79276,3.40
133529,2024-12-31 22:15:00,10226.0,10682.2425,0.0,80.2975,-6.26466,54.79276,3.40
133530,2024-12-31 22:30:00,10226.0,10533.0250,0.0,83.1350,-6.26466,54.79276,3.40
133531,2024-12-31 22:45:00,10226.0,10383.8075,0.0,85.9725,-6.26466,54.79276,3.40
